# 2. The R you need today

**Open:** the project, restart R, then Save As `scripts/my-02-minimum-r.R` from the supplied chapter script.

**Find at the end:** `outputs/practice/known-counts.csv`. The last section is a separate changed-copy exercise.

## 2.1 Warm up: a name, a function and a check

`c()` combines values into a vector. These practice counts are invented:

In [1]:
counts <- c(10, 20, 30)
counts

[1] 10 20 30

`mean(counts)` passes that vector to the `mean()` function. Parentheses hold its inputs, also called arguments.

In [2]:
mean(counts)
sum(counts) / length(counts)

[1] 20

[1] 20

**Check by hand:** sum 10, 20 and 30, then divide by three. Both expressions should give 20. R is case-sensitive: `counts` and `Counts` are different names.

## 2.2 Lab task: inspect four samples

A CSV is a table stored as text. Run `import` to read the included practice file:

In [3]:
practice <- read.csv("data/practice_counts.csv", stringsAsFactors = FALSE)
practice

sample_id,count_cfu_ml,storage_hours
<chr>,<int>,<int>
P01,10,0
P02,20,2
P03,30,4
P04,NA,6


One row is one invented sample; one column is one variable. The R object `practice` is a **data frame**, a table whose columns can hold different types of values. `count_cfu_ml` is the practice concentration in CFU/mL.

Run `structure`, then decode its output:

In [4]:
dim(practice)
names(practice)
str(practice)

[1] 4 3

[1] "sample_id"     "count_cfu_ml"  "storage_hours"

'data.frame':	4 obs. of  3 variables:
 $ sample_id    : chr  "P01" "P02" "P03" "P04"
 $ count_cfu_ml : int  10 20 30 NA
 $ storage_hours: int  0 2 4 6


| Output | What to check |
|---|---|
| `dim()` | Four rows and three columns, in that order |
| `names()` | The actual column names you must use |
| `str()` | Types: `int`/`num` are numbers; `chr` is text |

Now take one column. `$` means “this named column from this table”:

In [5]:
practice$count_cfu_ml
class(practice$count_cfu_ml)

[1] 10 20 30 NA

[1] "integer"

**Your turn:** display `practice$storage_hours`. Predict its four values before running. A sample ID labels a record; it is not automatically useful information for predicting its count.

## 2.3 Decision: what does the missing result mean?

The fourth count is `NA`: no recorded result. Run `missing`:

In [6]:
is.na(practice$count_cfu_ml)
sum(is.na(practice$count_cfu_ml))
mean(practice$count_cfu_ml)

[1] FALSE FALSE FALSE  TRUE

[1] 1

[1] NA

`is.na()` produces TRUE/FALSE flags. `sum()` counts each TRUE as 1, giving one missing entry. The ordinary mean is NA because a count is unknown.

**Question:** can we report a mean of the recorded counts while leaving that unknown measurement unchanged?

In [7]:
mean(practice$count_cfu_ml, na.rm = TRUE)

[1] 20

Yes: `na.rm = TRUE` omits missing entries **for this calculation**. The result is 20 for three recorded counts. It does not recover the fourth measurement. Replacing NA with zero would make a different assumption.

To keep only the known rows in a new object, run `subset`:

In [8]:
known_counts <- practice[!is.na(practice$count_cfu_ml), ]
known_counts

,sample_id,count_cfu_ml,storage_hours
,<chr>,<int>,<int>
1,P01,10,0
2,P02,20,2
3,P03,30,4


Read `[rows, columns]` as row and column selection. `!` means “not”; leaving the column position empty keeps all columns. The original `practice` remains available.

## 2.4 Evidence: save, reopen and check

**Supplied block — `export`.** Run it unchanged. It writes `known_counts` to a CSV, reads that file back, and checks that the values agree. You can expand the code for details.

In [9]:
dir.create("outputs/practice", recursive = TRUE, showWarnings = FALSE)
write.csv(known_counts, "outputs/practice/known-counts.csv", row.names = FALSE)
round_trip <- read.csv("outputs/practice/known-counts.csv")
isTRUE(all.equal(known_counts, round_trip, check.attributes = FALSE))

[1] TRUE

Find **Files → outputs → practice → known-counts.csv**. The read-back check should be TRUE. You need to explain this check's purpose, not reproduce its nested arguments from memory.

Before continuing, run the short `assertion` block:

In [10]:
stopifnot(nrow(practice) == 4)
stopifnot(sum(is.na(practice$count_cfu_ml)) == 1)
stopifnot(mean(known_counts$count_cfu_ml) == 20)
cat("Chapter 2 checks passed.\n")

Chapter 2 checks passed.


`stopifnot()` stops if a required condition is false. Passing conditions print nothing; the completion message shows that execution reached the end. `==` compares values, `<-` stores one, and `name = value` supplies a function setting.

```{admonition} Explain your result
:class: tip
Which three samples contribute to the mean of 20? Find the saved CSV and explain why the fourth count is not automatically zero.
```

**Ask AI:** provide the table and ask for a missing-count summary without imputation. Check its answer against the three recorded values. Chapter 3 takes this exchange through a full repair cycle.

## 2.5 Extension worksheet: change a copy

**Predict first:** replace the fourth count with 40 in a labeled practice copy. What will happen to the mean and missing count?

**Supplied block — `exercise-copy`.** It creates the changed copy below; the original input stays intact. Run the block unchanged:

In [11]:
# Extension exercise: write and read a NEW file; preserve the original input.
changed <- practice
changed$count_cfu_ml[4] <- 40
write.csv(changed, "outputs/practice/practice_counts_changed.csv", row.names = FALSE)

The next line explicitly reads the **new filename**. Another CSV on disk does not redirect an earlier import:

In [12]:
changed <- read.csv("outputs/practice/practice_counts_changed.csv")
data.frame(missing_n = sum(is.na(changed$count_cfu_ml)), mean_count = mean(changed$count_cfu_ml))

missing_n,mean_count
<int>,<dbl>
0,25


Now test the old statement, “one count is missing”:

In [13]:
old_rule_holds <- sum(is.na(changed$count_cfu_ml)) == 1
old_rule_holds

[1] FALSE

**FALSE is expected.** If you pass this false condition to `stopifnot()`, it raises an error. In this exercise the input was intentionally changed, so the old expectation should fail.

Run `exercise-check` to test the new expectations and verify that the original file still has its missing count:

In [14]:
stopifnot(sum(is.na(changed$count_cfu_ml)) == 0, mean(changed$count_cfu_ml) == 25)
original_again <- read.csv("data/practice_counts.csv")
stopifnot(is.na(original_again$count_cfu_ml[4]))
cat("Changed-copy exercise passed; original input preserved.\n")

Changed-copy exercise passed; original input preserved.


```{admonition} Open after you predict
:class: tip dropdown
The changed file is outputs/practice/practice_counts_changed.csv. It has no missing count and its mean is (10 + 20 + 30 + 40) / 4 = 25. The original data/practice_counts.csv still has its fourth count missing. In real data, investigate an unexplained failure before changing a check.
```

**Next:** [ask, run, inspect and repair](03-working-with-ai.ipynb). [Help with errors](../reference/troubleshooting.md).